In [1]:
import os
import uuid
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding, SparseTextEmbedding, LateInteractionTextEmbedding
from dotenv import load_dotenv
from utils.semantic_chuncker import SemanticChunker

load_dotenv()

/Users/luizfelipew/Documents/git/AI-Engineering/dev-eficiente-IA/engineering-ai/curso-ia/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
DENSE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SPARSE_MODEL = "Qdrant/bm25"
COLBERT_MODEL = "colbert-ir/colbertv2.0"
COLLECTION_NAME = "financial"
FILE_PATH = "./AAPL_10-K_1A_temp.md"
MAX_TOKENS = 300

qdrant = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
)


In [3]:
qdrant.delete_collection(COLLECTION_NAME)

True

In [4]:

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense":models.VectorParams (size=384,distance=models.Distance.COSINE),
        "colbert":models.VectorParams (
            size=128,
            distance=models.Distance.COSINE,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM
            )
        ),
    },
    sparse_vectors_config={"sparse": models.SparseVectorParams()}
)

True

In [5]:
from markdown_it.rules_block.paragraph import paragraph
with open(FILE_PATH, "r", encoding="utf-8") as f:
    content = f.read()
    
# paragraphs = content.split("\n\n")
# chunks = [p.strip() for p in paragraphs if len(p.strip()) > 50]

chunker = SemanticChunker(max_tokens=MAX_TOKENS)
chunks = chunker.create_chunks(content)



In [6]:
dense_model = TextEmbedding(DENSE_MODEL)
sparse_model = SparseTextEmbedding(SPARSE_MODEL)
colbert_model = LateInteractionTextEmbedding(COLBERT_MODEL)

points = []
for chunk in chunks:
    dense_embedding = list(dense_model.passage_embed([chunk]))[0].tolist()
    sparse_embedding = list(sparse_model.passage_embed([chunk]))[0].as_object()
    # ColBERT retorna múltiplos vetores (multivector) - precisa ser uma lista de listas
    colbert_vectors = list(colbert_model.passage_embed([chunk]))[0]
    colbert_embedding = [vec.tolist() for vec in colbert_vectors]
    
    point = models.PointStruct(
        id=str(uuid.uuid4()),
        vector={
            "dense": dense_embedding,
            "sparse": sparse_embedding,
            "colbert": colbert_embedding
        },
        payload={"text": chunk, "source": FILE_PATH},
    )
    points.append(point)

qdrant.upload_points(collection_name=COLLECTION_NAME, points=points)


Fetching 5 files: 100%|██████████| 5/5 [00:17<00:00,  3.48s/it]


In [7]:
query_text = "What are the main financial risks?"
query_dense = list(dense_model.query_embed([query_text]))[0].tolist()
query_sparse = list(sparse_model.query_embed([query_text]))[0].as_object()
query_colbert = list(colbert_model.query_embed([query_text]))[0].tolist()

results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        {
            "prefetch": [
                {"query": query_dense, "using": "dense", "limit": 10},
                {"query": query_sparse, "using": "sparse", "limit": 10},
            ],
            "query": models.FusionQuery(fusion=models.Fusion.RRF),
            "limit": 20
        }
    ],
    query=query_colbert,
    using="colbert",
    limit=3
)    
    

In [8]:
max_score = max(result.score for result in results.points)
for r in results.points:
    normalized_score = r.score / max_score
    print(f"Score: {normalized_score}")
    print(f"Texto: {r.payload['text'][:100]}...")
    print("-" * 80)

Score: 1.0
Texto: Item 1A. Risk Factors

The Company’s business, reputation, results of operations, financial conditio...
--------------------------------------------------------------------------------
Score: 0.9972171893287101
Texto: In addition to the risks generally relating to the collection, use, retention, security and transfer...
--------------------------------------------------------------------------------
Score: 0.9484136693480834
Texto: Investment in new business strategies and acquisitions could disrupt the Company’s ongoing business,...
--------------------------------------------------------------------------------
